In [1]:
%load_ext autoreload
%autoreload 2
#%env JAX_PLATFORM_NAME=gpu
#%env XLA_PYTHON_CLIENT_PREALLOCATE=false
import numpy as np
import numpy.linalg as lg
from morphomatics.manifold import Sphere, PowerManifold
from morphomatics.stats import ExponentialBarycenter
from timeseries.stats import sph_correlated_trjs
from timeseries.main import pred
from timeseries.model import Reg, RidgeReg, AVGEnsemble
from util_pred import cov_mat, fit_poly_dc

SEED      = 42
np.random.seed(SEED)

M  = Sphere()
Ex = ExponentialBarycenter()

DEGREE      = 5
N_SUBJ      = 30
N_TRAIN     = 20
N_VAL       = 5
N_TEST      = 5
N_POINTS    = 35
NOISE_STD   = 0.05
BETWEEN_STD = 2.0
LON_MAX     = 0.75 * np.pi
LAT_MAX     = np.pi / 20
DIAG_LOAD   = 0.0  #1e-6
TEMPLATES   = ['Geo', 'Poly', 'Else']
LAM_STAR    = [20.0, 11.0, 3.0]
LAG = True
AVG = AVGEnsemble(0.5)
PRED_ARGS = {'n_learn': DEGREE + 1, 'n_pred': 1, 'iterative': True}
print(f'seed={SEED}  degree={DEGREE}  split={N_TRAIN}/{N_VAL}/{N_TEST}')

seed=42  degree=5  split=20/5/5


## Helper functions

In [9]:
def save_path(template):
#    return f'synthetic_YB_{template}.npz'
    return f'../datasets/sph{template}.npz'

def save_data(Y, B, template):
    path = save_path(template)
    np.savez(path,
             Y=np.array(Y, dtype=object),
             B=np.array(B))
    print(f'  Saved -> {path}')

def load_data(template):
    path = save_path(template)
    data = np.load(path, allow_pickle=True)
    Y = list(data['Y'])
    B = list(data['B'])
    print(f'  Loaded <- {path}')
    return Y, B

def coarse_to_fine_grid_search(Y_val, model_fn, coarse_grid, n_refine=3, factor=4.0):
    """Two-stage coarse-to-fine grid search over lambda (ridge only)."""
    all_results = {}

    # Stage 1: coarse pass
    print(f'  Coarse pass: [{", ".join(f"{l:.1e}" for l in coarse_grid)}]')
    for lam in coarse_grid:
        _, m = pred(Y_val, model_fn(lam), **PRED_ARGS, prnt=False)
        all_results[lam] = m['mae']
        print(f'    lambda={lam:.1e}  val MAE={m["mae"]:.4f}')

    lam_coarse_best = min(all_results, key=all_results.get)

    # Stage 2: fine pass around best coarse lambda
    log_center = np.log10(lam_coarse_best)
    log_range  = np.log10(factor)
    fine_grid  = np.logspace(log_center - log_range, log_center + log_range, n_refine).tolist()
    fine_grid  = [l for l in fine_grid
                  if not any(abs(l - ev) / ev < 0.01 for ev in all_results)]

    if fine_grid:
        print(f'  Fine pass around lambda={lam_coarse_best:.1e}: [{", ".join(f"{l:.2e}" for l in fine_grid)}]')
        for lam in fine_grid:
            _, m = pred(Y_val, model_fn(lam), **PRED_ARGS, prnt=False)
            all_results[lam] = m['mae']
            print(f'    lambda={lam:.2e}  val MAE={m["mae"]:.4f}')

    return min(all_results, key=all_results.get), all_results

def run_experiment(template, Y_train, Y_val, Y_test, B_train):
    print(f"\n{'='*60}")
    print(f'Experiment: {template}')
    print(f"{'='*60}")

    n_cp = np.shape(B_train[0])[0]
    dim  = 3
    P    = PowerManifold(M, n_cp)
    PRED_ARGS['n_learn'] = n_cp
    # covariance — diagonal loading for numerical safety before cov_intrinsic
    mean_b  = Ex.compute(P, B_train, max_iter=30)
    cov_b   = cov_mat(P.metric.log, B_train, mean_b) + DIAG_LOAD * np.eye(n_cp * dim)
    eigvals = np.sort(lg.eigvalsh(cov_b))[::-1]
    print(f'  Eigenvalues: {np.round(eigvals, 6)}')

    # scale: noise^2 * M.dim / mean_dominant_eig
    # makes ridge_const = mu * scale, so mu=1 ~ theoretically balanced
    # dominant = above DIAG_LOAD (structural zeros lifted to DIAG_LOAD, still dominant below)
    # dominant eigenvalues: above numerical noise floor regardless of DIAG_LOAD
    dominant_eigs = eigvals[eigvals > 1e-6]
    scale_mu = 1e-3#NOISE_STD**2 * M.dim / np.mean(dominant_eigs)
    print(f'  scale_mu={scale_mu:.3e}  ({len(dominant_eigs)} dominant eigs, mean_eig={np.mean(dominant_eigs):.3e})')

    # model factory: ridge_const = mu * scale_mu passed to RidgeRegression
    def model_fn(lam):
        if lam == 0:
            return Reg(M, lag=LAG, degree=DEGREE)
        return RidgeReg(M, mean_b, cov_b, lam * scale_mu, lag=LAG, degree=DEGREE)

    # OLS on val and test
    _, m_ols_val = pred(Y_val,  model_fn(0), **PRED_ARGS, prnt=False)
    _, m_ols     = pred(Y_test, model_fn(0), **PRED_ARGS, prnt=False)
    print(f'  OLS  MAE: {m_ols["mae"]:.4f} +/- {m_ols["std"]:.4f}')
    if LAM_STAR is None:
        # grid search — centered around mu=1e-3..1 based on theoretical analysis
        # mu_balanced ~ mean_eig^2 * n_cp / (M.dim * dist_to_mean^2) ~ 1e-3
        COARSE_GRID = [0.2, 5, 10, 20]
        lam_star, val_results = coarse_to_fine_grid_search(
            Y_val, model_fn, COARSE_GRID, n_refine=3, factor=4.0
        )
        # fall back to OLS if ridge does not improve on val
        if m_ols_val['mae'] <= val_results[lam_star]:
            lam_star = 0
        print(f'  lambda* = {lam_star:.2e}  (ridge_const = {lam_star * scale_mu:.2e})')
    else:
        idx = TEMPLATES.index(template)
        lam_star = LAM_STAR[idx]
    # ridge on test
    _, m_ridge  = pred(Y_test, model_fn(lam_star), **PRED_ARGS, prnt=False)
    improvement = 100 * (m_ols['mae'] - m_ridge['mae']) / m_ols['mae']
    print(f'  Ridge MAE: {m_ridge["mae"]:.4f} +/- {m_ridge["std"]:.4f}  (improvement: {improvement:.1f}%)')
    return {
        'ols_mean':    m_ols['mae'],
        'ols_std':     m_ols['std'],
        'ridge_mean':  m_ridge['mae'],
        'ridge_std':   m_ridge['std'],
        'lam_star':    lam_star,
        'scale_mu':    scale_mu,
        'improvement': improvement,
    }


## Generate Data

Generates fresh data — **does not save to disk**.  
Inspect results first, then run the **Save** cell if satisfied.

In [8]:
all_data = {}
for template in TEMPLATES:
    print(f'Generating {N_SUBJ} trajectories (template={template}) ...')
    Y, _ = sph_correlated_trjs(
        LON_MAX, LAT_MAX,
        n_trj=N_SUBJ, n_points=N_POINTS,
        noise_std=NOISE_STD, between_std=BETWEEN_STD, mean_curve=template
    )
    print(f'  Fitting Bezier polynomials ...')
    B, _ = fit_poly_dc(M, Y, deg=DEGREE)
    all_data[template] = {'Y': Y, 'B': B}
    print(f'  Done.')
print("Data ready to run experiment. To save data run 'Save' cell.")

Generating 30 trajectories (template=Geo) ...
  Fitting Bezier polynomials ...
  Done.
Generating 30 trajectories (template=Poly) ...
  Fitting Bezier polynomials ...
  Done.
Generating 30 trajectories (template=Else) ...
  Fitting Bezier polynomials ...
  Done.
Data ready to run experiment. To save data run 'Save' cell.


## Load Data

Run this cell **instead of Generate** to reload previously saved data.

In [10]:
from timeseries.stats import load_sph
all_data = {}
for template in TEMPLATES:
    B, Y = load_sph(template)
    all_data[template] = {'Y': Y, 'B': B}

# Infer data parameters from loaded arrays
_Y0 = all_data[TEMPLATES[0]]['Y']
_B0 = all_data[TEMPLATES[0]]['B']
N_SUBJ   = len(_Y0)
N_POINTS = len(_Y0[0])
DEGREE   = len(_B0[0]) - 1
print(f'Loaded: N_SUBJ={N_SUBJ}, N_POINTS={N_POINTS}, DEGREE={DEGREE}')
print(f'Using split: {N_TRAIN}/{N_VAL}/{N_TEST}')


Loaded: N_SUBJ=30, N_POINTS=35, DEGREE=5
Using split: 20/5/5


## Run Experiments

In [11]:
results = {}
for template in TEMPLATES:
    d = all_data[template]
    Y, B = d['Y'], d['B']
    results[template] = run_experiment(
        template,
        Y[:N_TRAIN],
        Y[N_TRAIN:N_TRAIN + N_VAL],
        Y[N_TRAIN + N_VAL:],
        B[:N_TRAIN]
    )


Experiment: Geo
  Eigenvalues: [ 3.430e-03  2.294e-03  1.894e-03  1.361e-03  1.214e-03  9.910e-04
  4.640e-04  3.290e-04  1.870e-04  1.260e-04  1.010e-04  5.000e-05
  0.000e+00  0.000e+00  0.000e+00 -0.000e+00 -0.000e+00 -0.000e+00]
  scale_mu=1.000e-03  (12 dominant eigs, mean_eig=1.037e-03)
  OLS  MAE: 0.0310 TODO STD
  Ridge MAE: 0.0308 +/- TODO STD  (improvement: 0.7%)

Experiment: Poly
  Eigenvalues: [ 4.354e-03  2.919e-03  2.401e-03  1.794e-03  1.329e-03  8.450e-04
  4.430e-04  3.020e-04  2.380e-04  1.020e-04  7.100e-05  2.700e-05
  0.000e+00  0.000e+00  0.000e+00 -0.000e+00 -0.000e+00 -0.000e+00]
  scale_mu=1.000e-03  (12 dominant eigs, mean_eig=1.236e-03)
  OLS  MAE: 0.0360 TODO STD
  Ridge MAE: 0.0323 +/- TODO STD  (improvement: 10.3%)

Experiment: Else
  Eigenvalues: [ 6.021e-03  4.227e-03  3.463e-03  2.500e-03  1.486e-03  1.106e-03
  7.490e-04  5.640e-04  3.040e-04  1.850e-04  7.800e-05  3.600e-05
  0.000e+00 -0.000e+00 -0.000e+00 -0.000e+00 -0.000e+00 -0.000e+00]
  scale_m

## Table 1

In [ ]:
print(f"{'='*70}")
print('Table 1 -- MAE (radians), synthetic spherical trajectory forecasting')
print(f"{'='*70}")
print(f"{'Experiment':<12} {'OLS':<26} {'Ridge (proposed)':<26} {'Impr.':>6}  lambda*")
print('-'*70)
for name, r in results.items():
    print(f"{name:<12} "
          f"{r['ols_mean']:.4f} +/- {r['ols_std']:.4f}      "
          f"{r['ridge_mean']:.4f} +/- {r['ridge_std']:.4f}    "
          f"{r['improvement']:+.1f}%   {r['lam_star']}")
print('-'*70)
avg_imp = np.mean([r['improvement'] for r in results.values()])
print(f"{'Average improvement:':<40} {avg_imp:+.1f}%")
print(f"{'='*70}")
print(f'seed={SEED}  |  degree={DEGREE}  |  split={N_TRAIN}/{N_VAL}/{N_TEST}')
print(f'covariance: sample + {DIAG_LOAD}*I (diagonal loading)')

## Save Data

Run this cell **only when satisfied** with the results.

In [ ]:
for template in TEMPLATES:
    d = all_data[template]
    save_data(d['Y'], d['B'], template)
print('All data saved.')